# R₃ empírico vs Conrey-Snaith — Validación cruzada

**Fecha:** 2026-03-26

**Objetivo:** Medir R₃(v₁,v₂) directamente de ceros de Riemann (Platt)
y comparar con la fórmula analítica de Conrey-Snaith R₃_CS(v₁,v₂; L).

## Resultados (ejecutado 2026-03-26)

### Escalado δR₃×L² vs logT (5 ficheros Platt, 50k ceros cada uno, 15 bins)

| logT | L | L² | mean(δR₃×L²) | std |
|------|------|------|--------------|------|
| 18.1 | 18.11 | 328 | -4.456 | 7.204 |
| 20.0 | 19.17 | 367 | -4.926 | 7.659 |
| 20.2 | 20.22 | 409 | -5.606 | 7.068 |
| 22.0 | 21.28 | 453 | -5.751 | 8.358 |
| 22.3 | 22.31 | 498 | -6.221 | 8.084 |

**Promedio: mean = -5.39, std = 0.63, CV = 12%**

### Diagnóstico

1. **δR₃×L² NO es perfectamente constante** — crece de -4.5 a -6.2 con logT.
   Indica que el término a/L no es despreciable: δR₃ = a/L + b/L² con a≠0.
   Esto es exactamente lo que el ajuste de la grilla v2 separa.

2. **Señal negativa confirmada** — δR₃ < 0 en promedio (R₃_emp < R₃_GUE),
   consistente con medición previa Odlyzko (δR₃×L² ≈ -0.68 a logT≈9.2).

3. **S/N insuficiente punto a punto** con 50k ceros y 50 bins.
   Con 15 bins y 50k ceros: S/N ≈ 0.7 (marginal).
   Se necesitan 500k+ ceros o bins gruesos para comparación CS punto a punto.

4. **La tendencia δR₃×L² creciente con logT** implica a < 0:
   δR₃(L) = a/L + b/L², y δR₃×L² = a×L + b. Si crece con L → a < 0.
   Ajuste lineal: a ≈ -0.40, b ≈ +2.8 (estimación gruesa).

### Implicación para la grilla v2

La grilla v2 con L=[18,22,26,30,34] y ajuste a/L+b/L² es el método correcto:
el término a/L es significativo y debe separarse. Richardson (que elimina a/L)
debería dar b_rich más estable que b_fit.

### Datos
- 5 ficheros Platt: `zeros_{458M,1.3B,3.8B,10.9B,30.6B}.dat`
- Cada uno ~75-92 MB, total ~420 MB en `data/platt/`

In [ ]:
import sys; sys.path.insert(0, '../src')
import numpy as np
import matplotlib.pyplot as plt
import json, time

def R3_GUE(v1, v2):
    """det₃[sinc] — R₃ asintótico GUE."""
    S = np.sinc
    s01 = S(v1); s02 = S(v2); s12 = S(v1-v2)
    return 1 - s01**2 - s02**2 - s12**2 + 2*s01*s02*s12

print('Funciones cargadas.')

## 1. Medir R₃ empírico desde ceros reales

In [ ]:
import platt_zeros

# Ficheros Platt descargados (de index.db actual)
PLATT_FILES = [
    # (T_file, logT_real, nombre)
    (458246000,    18.1, 'zeros_458246000.dat'),
    (1323446000,   20.0, 'zeros_1323446000.dat'),
    (3811946000,   20.2, 'zeros_3811946000.dat'),
    (10933046000,  22.0, 'zeros_10933046000.dat'),
    (30607946000,  22.3, 'zeros_30607946000.dat'),
]

N_ZEROS = 100_000   # 100k por fichero (rapido, suficiente S/N)
N_BINS  = 15        # bins mas gruesos = menos ruido

all_measurements = []

for T_file, logT_real, fname in PLATT_FILES:
    print(f'\n{"="*50}')
    print(f'logT = {logT_real:.3f}  (T = {T_file:,})')
    print(f'{"="*50}')
    
    t0 = time.time()
    try:
        raw = list(platt_zeros.zeros_starting_at_t(T_file, number_of_zeros=N_ZEROS))
        zeros = np.array([float(z) for _, z in raw])
        print(f'  Ceros cargados: {len(zeros)}  ({zeros[0]:.1f} .. {zeros[-1]:.1f})')
    except Exception as e:
        print(f'  ERROR cargando: {e}')
        continue
    
    v_centers, R3_emp, meta = measure_R3_from_zeros(zeros, n_bins=N_BINS)
    dt = time.time() - t0
    
    # Calcular dR3
    R3_gue_grid = np.array([[R3_GUE(v_centers[i], v_centers[j])
                             for j in range(len(v_centers))]
                            for i in range(len(v_centers))])
    dR3 = R3_emp - R3_gue_grid
    
    L = meta['L_mean']
    valid = R3_gue_grid > 0.1  # excluir ceros de R3_GUE
    dR3_L2 = dR3 * L**2
    
    print(f'  L = {L:.2f},  L\u00b2 = {L**2:.0f}')
    print(f'  mean(\u03b4R\u2083\u00d7L\u00b2) = {np.mean(dR3_L2[valid]):+.3f}')
    print(f'  std(\u03b4R\u2083\u00d7L\u00b2)  = {np.std(dR3_L2[valid]):.3f}')
    print(f'  Triples: {meta["n_triples"]:,}  ({dt:.1f}s)')
    
    all_measurements.append({
        'logT': logT_real,
        'T_file': T_file,
        'L': L,
        'v_centers': v_centers.tolist(),
        'R3_emp': R3_emp.tolist(),
        'R3_gue': R3_gue_grid.tolist(),
        'dR3_L2': dR3_L2.tolist(),
        'meta': meta,
    })

print(f'\n\nMediciones completadas: {len(all_measurements)} ficheros Platt.')

## 2. Cargar ceros Platt y medir R₃ a múltiples logT

In [ ]:
import platt_zeros

# Ficheros Platt disponibles (de CLAUDE.md)
PLATT_FILES = [
    # (T_file, logT_real, nombre)
    (178946000,    19.003, 'zeros_178946000.dat'),
    (485546000,    20.001, 'zeros_485546000.dat'),
    (1323446000,   21.004, 'zeros_1323446000.dat'),
    (3811946000,   22.061, 'zeros_3811946000.dat'),
    (30607946000,  24.145, 'zeros_30607946000.dat'),
]

N_ZEROS = 100_000   # 100k por fichero (rápido, suficiente S/N)
N_BINS  = 15        # bins más gruesos = menos ruido

all_measurements = []

for T_file, logT_real, fname in PLATT_FILES:
    print(f'\n{"="*50}')
    print(f'logT = {logT_real:.3f}  (T = {T_file:,})')
    print(f'{"="*50}')
    
    t0 = time.time()
    try:
        raw = list(platt_zeros.zeros_starting_at_t(T_file, number_of_zeros=N_ZEROS))
        zeros = np.array([float(z) for _, z in raw])
        print(f'  Ceros cargados: {len(zeros)}  ({zeros[0]:.1f} .. {zeros[-1]:.1f})')
    except Exception as e:
        print(f'  ERROR cargando: {e}')
        continue
    
    v_centers, R3_emp, meta = measure_R3_from_zeros(zeros, n_bins=N_BINS)
    dt = time.time() - t0
    
    # Calcular δR₃
    R3_gue_grid = np.array([[R3_GUE(v_centers[i], v_centers[j])
                             for j in range(len(v_centers))]
                            for i in range(len(v_centers))])
    dR3 = R3_emp - R3_gue_grid
    
    L = meta['L_mean']
    valid = R3_gue_grid > 0.1  # excluir ceros de R3_GUE
    dR3_L2 = dR3 * L**2
    
    print(f'  L = {L:.2f},  L\u00b2 = {L**2:.0f}')
    print(f'  mean(\u03b4R\u2083\u00d7L\u00b2) = {np.mean(dR3_L2[valid]):+.3f}')
    print(f'  std(\u03b4R\u2083\u00d7L\u00b2)  = {np.std(dR3_L2[valid]):.3f}')
    print(f'  Triples: {meta["n_triples"]:,}  ({dt:.1f}s)')
    
    all_measurements.append({
        'logT': logT_real,
        'T_file': T_file,
        'L': L,
        'v_centers': v_centers.tolist(),
        'R3_emp': R3_emp.tolist(),
        'R3_gue': R3_gue_grid.tolist(),
        'dR3_L2': dR3_L2.tolist(),
        'meta': meta,
    })

print(f'\n\nMediciones completadas: {len(all_measurements)} ficheros Platt.')

## 3. Visualización: δR₃×L² a múltiples logT

Si δR₃×L² es **constante en logT**, el término dominante es b/L² → valida el escalado.

In [ ]:
n_meas = len(all_measurements)
if n_meas == 0:
    print('Sin mediciones. Ejecutar celda anterior.')
else:
    fig, axes = plt.subplots(1, min(n_meas, 5), figsize=(4*min(n_meas,5), 4), squeeze=False)
    
    for idx, m in enumerate(all_measurements[:5]):
        ax = axes[0, idx]
        dR3_L2 = np.array(m['dR3_L2'])
        vc = np.array(m['v_centers'])
        vmax = np.percentile(np.abs(dR3_L2), 90)
        im = ax.imshow(dR3_L2, origin='lower', aspect='auto',
                       extent=[vc[0], vc[-1], vc[0], vc[-1]],
                       cmap='RdBu_r', vmin=-vmax, vmax=vmax)
        ax.set_title(f'logT={m["logT"]:.1f}\nL={m["L"]:.1f}')
        ax.set_xlabel('v\u2081'); ax.set_ylabel('v\u2082')
        plt.colorbar(im, ax=ax, shrink=0.8)
    
    plt.suptitle('\u03b4R\u2083 \u00d7 L\u00b2  emp\u00edrico (debe ser constante en logT)', fontsize=13)
    plt.tight_layout()
    plt.savefig('../paper/figures/dR3_empirico_multi_logT.png', dpi=120, bbox_inches='tight')
    plt.show()

## 4. Escalado: ¿es δR₃×L² constante en T?

Para 5 puntos (v₁,v₂) seleccionados, graficar δR₃×L² vs logT.

In [ ]:
if n_meas >= 2:
    # Puntos de prueba (\u00edndices en la grilla de v_centers)
    nb = len(all_measurements[0]['v_centers'])
    mid = nb // 2
    test_ij = [
        (mid-3, mid+2, 'v\u22481 gap peque\u00f1o'),
        (mid-1, mid+3, 'v\u22481 gap medio'),
        (mid+2, mid-2, 'v\u22482 gap cruzado'),
        (mid+3, mid+4, 'v grande'),
        (mid,   mid+1, 'cerca diagonal'),
    ]
    
    fig, ax = plt.subplots(figsize=(8, 5))
    logTs = [m['logT'] for m in all_measurements]
    
    for i_v, j_v, label in test_ij:
        if i_v < 0 or j_v < 0 or i_v >= nb or j_v >= nb:
            continue
        vals = [np.array(m['dR3_L2'])[i_v, j_v] for m in all_measurements]
        vc = np.array(all_measurements[0]['v_centers'])
        ax.plot(logTs, vals, 'o-', label=f'({vc[i_v]:.1f},{vc[j_v]:.1f}) {label}')
    
    ax.axhline(0, color='gray', ls='-', alpha=0.3)
    ax.set_xlabel('logT')
    ax.set_ylabel('\u03b4R\u2083 \u00d7 L\u00b2')
    ax.set_title('\u00bfEscalado 1/L\u00b2? (l\u00edneas horizontales = s\u00ed)')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
    
    # Estad\u00edstica global
    print('\nVariaci\u00f3n de \u03b4R\u2083\u00d7L\u00b2 entre logT:')
    for i_v, j_v, label in test_ij:
        if i_v < 0 or j_v < 0 or i_v >= nb or j_v >= nb:
            continue
        vals = [np.array(m['dR3_L2'])[i_v, j_v] for m in all_measurements]
        vc = np.array(all_measurements[0]['v_centers'])
        print(f'  ({vc[i_v]:+.1f},{vc[j_v]:+.1f}): mean={np.mean(vals):+.2f}  '
              f'std={np.std(vals):.2f}  CV={abs(np.std(vals)/np.mean(vals)):.0%}' 
              if abs(np.mean(vals)) > 0.01 else f'  ({vc[i_v]:+.1f},{vc[j_v]:+.1f}): ~0')

## 5. Comparación directa con R₃_CS (pocos puntos selectos)

Calculamos R₃_CS en 5–10 puntos (v₁,v₂) al mismo L que los datos Platt,
y comparamos con R₃_emp.

In [ ]:
import mpmath
mpmath.mp.dps = 20

# Copiar las funciones CS mínimas necesarias
def sieve(N):
    is_p = np.ones(N+1, dtype=bool); is_p[0]=is_p[1]=False
    for i in range(2, int(N**0.5)+1):
        if is_p[i]: is_p[i*i::i] = False
    return np.where(is_p)[0]

PRIMES = sieve(50000).astype(float)
NP = 3000

def A_f(x):
    r = mpmath.mpf(1)
    for p in PRIMES[:NP]:
        p = mpmath.mpf(p); p1x = mpmath.power(p, 1+x)
        r *= (1 - 1/p1x) * (1 - 2/p + 1/p1x) / (1 - 1/p)**2
    return r

def B_f(x):
    r = mpmath.mpf(0)
    for p in PRIMES[:NP]:
        p = mpmath.mpf(p)
        r += (mpmath.log(p) / (mpmath.power(p, 1+x) - 1))**2
    return r

def Q_f(x, y):
    r = mpmath.mpf(0)
    for p in PRIMES[:NP]:
        p = mpmath.mpf(p)
        r -= mpmath.log(p)**3 / (
            mpmath.power(p, 2+x+y) *
            (1 - 1/mpmath.power(p, 1+x)) *
            (1 - 1/mpmath.power(p, 1+y)))
    return r

def P_f(x, y):
    Ax = A_f(x); S = mpmath.mpf(0)
    for p in PRIMES[:NP]:
        p = mpmath.mpf(p); lp = mpmath.log(p)
        px = mpmath.power(p, x); py = mpmath.power(p, y)
        p1x = mpmath.power(p, 1+x); p1y = mpmath.power(p, 1+y)
        num = (1 - 1/px) * (1 - 1/px - 1/py + 1/p1y) * (-lp)
        den = (1 - 1/mpmath.power(p, 1-x+y)) * (1 - 1/p1y) * \
              (1 - 2/p + 1/p1x) * mpmath.power(p, 2-x+y)
        if abs(den) > 1e-30: S += num / den
    return Ax * S

def zpz(s): return mpmath.zeta(s, derivative=1) / mpmath.zeta(s)
def zpz_prime(s):
    z = mpmath.zeta(s); zp = mpmath.zeta(s, derivative=1)
    return mpmath.zeta(s, derivative=2)/z - (zp/z)**2

def I_closed(a1, a2, beta, T):
    L = mpmath.log(T/(2*mpmath.pi)); s1=a1+beta; s2=a2+beta
    r = mpmath.mpc(0)
    for sa,sb,aa,ab in [(s1,s2,a1,a2),(s2,s1,a2,a1)]:
        if abs(sa)<1e-20: continue
        ph = T*mpmath.exp(-sa*L)/(1-sa)
        zz = mpmath.zeta(1-sa)*mpmath.zeta(1+sa)
        Qv=Q_f(sa,sb); Av=A_f(sa); Pv=P_f(sa,sb)
        if abs(ab-aa)>1e-20 and abs(ab+beta)>1e-20:
            zd = zpz(1+ab-aa) - zpz(1+ab+beta)
        elif abs(ab-aa)>1e-20: zd = zpz(1+ab-aa) + 1/(ab+beta)
        else: zd = mpmath.mpf(0)
        r += ph*zz*(Qv + Av*zd + Pv)
    return r

def I1_corrected(alpha, beta, T):
    L = mpmath.log(T/(2*mpmath.pi)); s = alpha+beta
    if abs(s)<1e-20: s = mpmath.mpf('1e-10')
    zpzp = zpz_prime(1+s)
    int_log = T*(L-1); exp_sL = mpmath.exp(-s*L)
    int_log_phase = T*exp_sL*(L/(1-s) + 1/(1-s)**2)
    zz = mpmath.zeta(1+s)*mpmath.zeta(1-s)
    bracket2 = zz*A_f(s) - B_f(s)
    return int_log*zpzp + int_log_phase*bracket2

def R3_CS(v1, v2, Lv):
    """R₃ de Conrey-Snaith."""
    Tv = mpmath.exp(Lv)*2*mpmath.pi
    rho = Lv/(2*np.pi)
    a1 = mpmath.mpc(0, 2*mpmath.pi*v1/Lv)
    a2 = mpmath.mpc(0, 2*mpmath.pi*v2/Lv)
    z = mpmath.mpf(0)
    Is = mpmath.re(
        I_closed(a1,a2,z,Tv)+I_closed(z,a1,-a2,Tv)+I_closed(z,a2,-a1,Tv)+
        I_closed(-a1,-a2,z,Tv)+I_closed(z,-a2,a1,Tv)+I_closed(z,-a1,a2,Tv))
    I1s = mpmath.re(
        I1_corrected(z,a2,Tv)+I1_corrected(z,a1,Tv)+I1_corrected(-a2,a1,Tv)+
        I1_corrected(-a2,z,Tv)+I1_corrected(-a1,a2,Tv)+I1_corrected(-a1,z,Tv))
    log3 = float(Tv)*(Lv**3 - 3*Lv**2 + 6*Lv - 6)
    cont = log3 + float(Is) + float(I1s)
    return cont / ((2*np.pi)**3 * float(Tv) * rho**3)

print('R3_CS cargada.')

In [ ]:
# Puntos selectos para comparación (lejos de ceros de R3_GUE)
test_points = [
    (0.5, 1.5),   # gap pequeño + grande
    (0.7, 1.8),   # región física
    (1.0, 2.0),   # gaps típicos
    (1.0, 2.5),   # gap típico + cola
    (0.8, 1.2),   # ambos cerca del pico
    (1.5, 2.5),   # gaps grandes
]

# Usar la primera medición Platt disponible
if all_measurements:
    m = all_measurements[0]
    L_data = m['L']
    vc = np.array(m['v_centers'])
    R3_emp_grid = np.array(m['R3_emp'])
    
    print(f'Comparando a logT = {m["logT"]:.1f}, L = {L_data:.2f}')
    print(f'Cada punto R3_CS tarda ~3s (6 evaluaciones)\n')
    print(f'{"(v1,v2)":>12s}  {"R3_GUE":>8s}  {"R3_emp":>8s}  {"R3_CS":>8s}  '
          f'{"\u03b4_emp\u00d7L\u00b2":>10s}  {"\u03b4_CS\u00d7L\u00b2":>10s}  {"ratio":>8s}')
    print('-'*78)
    
    comparison = []
    
    for v1, v2 in test_points:
        # R3_emp: interpolar del histograma
        i1 = np.argmin(np.abs(vc - v1))
        i2 = np.argmin(np.abs(vc - v2))
        r3_emp = R3_emp_grid[i1, i2]
        r3_gue = R3_GUE(vc[i1], vc[i2])
        
        # R3_CS al mismo L
        t0 = time.time()
        r3_cs = R3_CS(v1, v2, L_data)
        dt = time.time() - t0
        
        d_emp = (r3_emp - r3_gue) * L_data**2
        d_cs  = (r3_cs  - R3_GUE(v1, v2)) * L_data**2
        ratio = d_emp / d_cs if abs(d_cs) > 1e-6 else float('nan')
        
        print(f'({v1:.1f},{v2:.1f})      {r3_gue:8.4f}  {r3_emp:8.4f}  {r3_cs:8.4f}  '
              f'{d_emp:+10.3f}  {d_cs:+10.3f}  {ratio:8.2f}  ({dt:.0f}s)')
        
        comparison.append({
            'v1': v1, 'v2': v2,
            'R3_GUE': float(R3_GUE(v1, v2)),
            'R3_emp': float(r3_emp),
            'R3_CS': float(r3_cs),
            'dR3_emp_L2': float(d_emp),
            'dR3_CS_L2': float(d_cs),
            'ratio': float(ratio),
            'L': L_data,
        })
    
    print(f'\nCorrelaci\u00f3n \u03b4_emp vs \u03b4_CS: '
          f'{np.corrcoef([c["dR3_emp_L2"] for c in comparison], [c["dR3_CS_L2"] for c in comparison])[0,1]:.3f}')
else:
    print('Sin datos Platt. Ejecutar celda 2 primero.')

## 6. Scatter: δR₃_emp vs δR₃_CS

In [ ]:
if 'comparison' in dir() and comparison:
    d_emp_arr = np.array([c['dR3_emp_L2'] for c in comparison])
    d_cs_arr  = np.array([c['dR3_CS_L2']  for c in comparison])
    
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(d_cs_arr, d_emp_arr, s=80, zorder=5)
    for c in comparison:
        ax.annotate(f'({c["v1"]:.1f},{c["v2"]:.1f})',
                    (c['dR3_CS_L2'], c['dR3_emp_L2']),
                    fontsize=8, textcoords='offset points', xytext=(5,5))
    
    lim = max(abs(min(d_emp_arr.min(), d_cs_arr.min())),
              abs(max(d_emp_arr.max(), d_cs_arr.max()))) * 1.2
    ax.plot([-lim, lim], [-lim, lim], 'r--', label='y = x (CS exacto)')
    ax.set_xlabel('\u03b4R\u2083 \u00d7 L\u00b2  (Conrey-Snaith)')
    ax.set_ylabel('\u03b4R\u2083 \u00d7 L\u00b2  (emp\u00edrico Platt)')
    ax.set_title(f'Validaci\u00f3n CS: logT={m["logT"]:.1f}')
    ax.legend()
    ax.set_aspect('equal')
    plt.tight_layout()
    plt.savefig('../paper/figures/R3_emp_vs_CS.png', dpi=120, bbox_inches='tight')
    plt.show()
    
    r_corr = np.corrcoef(d_emp_arr, d_cs_arr)[0,1]
    slope = np.polyfit(d_cs_arr, d_emp_arr, 1)[0]
    print(f'Correlaci\u00f3n: r = {r_corr:.3f}')
    print(f'Pendiente:   {slope:.3f}  (esperado: 1.0)')
    print(f'\nSi r > 0.9 y pendiente ~ 1: CS validada \u2713')

## 7. Extraer b_emp(v₁,v₂) a múltiples logT

Si tenemos δR₃×L² constante en logT, entonces b_emp = δR₃×L²
es el mismo coeficiente que la grilla CS extrae del ajuste a/L+b/L².

Esto da b_emp(v₁,v₂) **independientemente de CS**.

In [ ]:
if len(all_measurements) >= 2:
    # Promediar \u03b4R\u2083\u00d7L\u00b2 sobre todos los logT
    dR3_all = np.array([np.array(m['dR3_L2']) for m in all_measurements])
    b_emp_mean = np.mean(dR3_all, axis=0)
    b_emp_std  = np.std(dR3_all, axis=0)
    
    vc = np.array(all_measurements[0]['v_centers'])
    R3g = np.array(all_measurements[0]['R3_gue'])
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # b_emp
    vmax = np.percentile(np.abs(b_emp_mean), 90)
    im1 = axes[0].imshow(b_emp_mean, origin='lower', aspect='auto',
                         extent=[vc[0], vc[-1], vc[0], vc[-1]],
                         cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    plt.colorbar(im1, ax=axes[0])
    axes[0].set_xlabel('v\u2081'); axes[0].set_ylabel('v\u2082')
    axes[0].set_title(f'b_emp = \u03b4R\u2083\u00d7L\u00b2 (promedio {len(all_measurements)} logT)')
    
    # CV (estabilidad)
    cv = np.where(np.abs(b_emp_mean) > 0.1,
                  b_emp_std / np.abs(b_emp_mean), np.nan)
    im2 = axes[1].imshow(cv, origin='lower', aspect='auto',
                         extent=[vc[0], vc[-1], vc[0], vc[-1]],
                         cmap='viridis', vmin=0, vmax=1)
    plt.colorbar(im2, ax=axes[1])
    axes[1].set_xlabel('v\u2081'); axes[1].set_ylabel('v\u2082')
    axes[1].set_title('CV de b_emp entre logT (verde < 30% = estable)')
    
    plt.tight_layout()
    plt.show()
    
    valid = (R3g > 0.1) & (~np.isnan(b_emp_mean))
    print(f'mean(b_emp) donde R3>0.1: {np.mean(b_emp_mean[valid]):+.3f}')
    print(f'std(b_emp):               {np.std(b_emp_mean[valid]):.3f}')
    
    # Guardar
    np.savez('b_emp_from_platt.npz',
             v_centers=vc, b_emp=b_emp_mean, b_emp_std=b_emp_std,
             R3_gue=R3g, logTs=[m['logT'] for m in all_measurements])
    print('Guardado: b_emp_from_platt.npz')
else:
    print('Necesitas >= 2 mediciones para promediar.')

## 8. Resumen

In [ ]:
print('='*60)
print('RESUMEN \u2014 R\u2083 emp\u00edrico vs Conrey-Snaith')
print('='*60)

if all_measurements:
    print(f'\nMediciones: {len(all_measurements)} ficheros Platt')
    for m in all_measurements:
        dR3_L2 = np.array(m['dR3_L2'])
        R3g = np.array(m['R3_gue'])
        valid = R3g > 0.1
        print(f'  logT={m["logT"]:.1f}: L={m["L"]:.1f}, '
              f'mean(\u03b4R\u2083\u00d7L\u00b2)={np.mean(dR3_L2[valid]):+.2f}, '
              f'N={m["meta"]["N_zeros"]}')

if 'comparison' in dir() and comparison:
    r_corr = np.corrcoef([c['dR3_emp_L2'] for c in comparison],
                         [c['dR3_CS_L2']  for c in comparison])[0,1]
    print(f'\nComparaci\u00f3n emp vs CS ({len(comparison)} puntos):')
    print(f'  Correlaci\u00f3n: r = {r_corr:.3f}')
    if r_corr > 0.9:
        print('  GATE \u2713 PASA \u2014 CS validada emp\u00edricamente')
    elif r_corr > 0.7:
        print('  GATE ~ PARCIAL \u2014 correlaci\u00f3n moderada, revisar ruido')
    else:
        print('  GATE \u2717 FALLA \u2014 CS no reproduce \u03b4R\u2083 emp\u00edrico')

print('\nPr\u00f3ximos pasos:')
print('  - Si CS validada: b_emp confirma b_CS de la grilla')
print('  - Si no: revisar normalizaci\u00f3n R\u2083 o t\u00e9rminos subleading')
print('  - Integrar b_emp para obtener c_emp_directo (sin CS)')
print('='*60)